In [ ]:
import pdfplumber
import pandas as pd

# Test that everything loads
print("Libraries loaded successfully")
print("Arpo data pipeline ready")

In [4]:
import pdfplumber

# Load the Fortune February PDF
with pdfplumber.open("../data/fortune 15kg febraury sales statement.pdf") as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        print(text)
        print("---PAGE BREAK---")

KAMADHENU ENTERPRISES
#69, 2ND MAIN ROAD , NEW THARUGUPET,,
BENGALURU : 560 002
Ph: 080 26700951 \ 080 26702002
FORTUNE R.OIL 15KG TIN
Stock Item Register
1-Feb-26 to 28-Feb-26
Page 1
Date Particulars Vch Type Vch No. Inwards Outwards Closing
Quantity Value Quantity Value Quantity Value
1-Feb-26 Opening Balance 56 T 1,28,163.24 56 T 1,28,163.24
1-Feb-26 Cash Sales Cash Sales K/CS/16899 1 T 2,523.81 55 T 1,25,874.61
2-Feb-26 Cash Sales Cash Sales K/CS/16907 1 T 2,523.81
KONARK VEG RESTAURANT Credit Sales K/CR/3308 13 T 33,738.12
SREE NARASIMHA HOTEL PVT LTD-2 Credit Sales K/CR/3309 10 T 25,952.40
KARNATAKA OIL STORES Credit Sales K/CR/3321 40 T 1,02,285.60 (-)9 T (-)20,597.66
3-Feb-26 Cash Sales Cash Sales K/CS/16954 1 T 2,561.90
Sri Padmashree Trading Company Purchase 5605 23 T 56,514.22
Sri Padmashree Trading Company Purchase 5616 70 T 1,78,666.60 83 T 1,90,248.30
4-Feb-26 Sangam Sweets Credit Sales K/CR/3333 10 T 25,952.40
SRI SIDDIVINAYAKA VENTURES {VEG FEAST} Credit Sales K/CR/3340

In [6]:
import pdfplumber
import pandas as pd
import re

def parse_tally_pdf(filepath):
    rows = []
    current_date = None
    
    with pdfplumber.open(filepath) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if not text:
                continue
            
            for line in text.split('\n'):
                # Match date at start of line
                date_match = re.match(r'^(\d{1,2}-\w{3}-\d{2})', line)
                if date_match:
                    current_date = date_match.group(1)
                
                # Skip header/footer lines
                if any(skip in line for skip in ['Carried Over', 'Brought Forward', 
                       'Totals', 'Opening Balance', 'Page', 'Date', 'KAMADHENU',
                       'Stock Item', 'Quantity', 'FORTUNE', 'RUCHI', 'Ph:']):
                    continue
                
                # Match transaction lines
                if current_date and ('Cash Sales' in line or 'Credit Sales' in line or 
                                     'Purchase' in line or 'Credit Note' in line):
                    
                    # Determine transaction type
                    if 'Purchase' in line:
                        txn_type = 'Purchase'
                    elif 'Credit Note' in line:
                        txn_type = 'Credit Note'
                    elif 'Credit Sales' in line:
                        txn_type = 'Credit Sale'
                    else:
                        txn_type = 'Cash Sale'
                    
                    # Extract customer name
                    if 'Cash Sales' in line:
                        customer = 'Cash Customer'
                    else:
                        customer = line.split('Credit Sales')[0].split('Purchase')[0].split('Credit Note')[0].strip()
                        if not customer:
                            customer = 'Unknown'
                    
                    # Extract numbers from line
                    numbers = re.findall(r'\(?\-?\d+(?:,\d+)*(?:\.\d+)?\)?', line)
                    numbers = [n.replace(',', '').replace('(', '-').replace(')', '') for n in numbers]
                    numbers = [float(n) for n in numbers if n not in ['', '-']]
                    
                    rows.append({
                        'date': current_date,
                        'customer': customer,
                        'type': txn_type,
                        'raw_numbers': numbers,
                        'raw_line': line.strip()
                    })
    
    return pd.DataFrame(rows)

# Parse Fortune February
df = parse_tally_pdf("../data/fortune 15kg febraury sales statement.pdf")
print(f"Total rows extracted: {len(df)}")
print(df[['date', 'customer', 'type']].head(20))

Total rows extracted: 72
        date                                customer         type
0   1-Feb-26                           Cash Customer    Cash Sale
1   2-Feb-26                           Cash Customer    Cash Sale
2   2-Feb-26                   KONARK VEG RESTAURANT  Credit Sale
3   2-Feb-26          SREE NARASIMHA HOTEL PVT LTD-2  Credit Sale
4   2-Feb-26                    KARNATAKA OIL STORES  Credit Sale
5   3-Feb-26                           Cash Customer    Cash Sale
6   3-Feb-26          Sri Padmashree Trading Company     Purchase
7   3-Feb-26          Sri Padmashree Trading Company     Purchase
8   4-Feb-26                  4-Feb-26 Sangam Sweets  Credit Sale
9   4-Feb-26  SRI SIDDIVINAYAKA VENTURES {VEG FEAST}  Credit Sale
10  4-Feb-26          Sri Padmashree Trading Company     Purchase
11  5-Feb-26                           Cash Customer    Cash Sale
12  5-Feb-26                           Cash Customer    Cash Sale
13  6-Feb-26              6-Feb-26 SHREE MILK SUPPL

In [7]:
# Fix customer name cleanup
df['customer'] = df['customer'].apply(lambda x: re.sub(r'^\d{1,2}-\w{3}-\d{2}\s*', '', x).strip())

# Check fix worked
print(df[['date', 'customer', 'type']].head(20))

        date                                customer         type
0   1-Feb-26                           Cash Customer    Cash Sale
1   2-Feb-26                           Cash Customer    Cash Sale
2   2-Feb-26                   KONARK VEG RESTAURANT  Credit Sale
3   2-Feb-26          SREE NARASIMHA HOTEL PVT LTD-2  Credit Sale
4   2-Feb-26                    KARNATAKA OIL STORES  Credit Sale
5   3-Feb-26                           Cash Customer    Cash Sale
6   3-Feb-26          Sri Padmashree Trading Company     Purchase
7   3-Feb-26          Sri Padmashree Trading Company     Purchase
8   4-Feb-26                           Sangam Sweets  Credit Sale
9   4-Feb-26  SRI SIDDIVINAYAKA VENTURES {VEG FEAST}  Credit Sale
10  4-Feb-26          Sri Padmashree Trading Company     Purchase
11  5-Feb-26                           Cash Customer    Cash Sale
12  5-Feb-26                           Cash Customer    Cash Sale
13  6-Feb-26                       SHREE MILK SUPPLY  Credit Sale
14  6-Feb-

In [9]:
# Parse all 3 files
fortune_feb = parse_tally_pdf("../data/fortune 15kg febraury sales statement.pdf")
ruchi_jan = parse_tally_pdf("../data/ruchi 15kg po tin january statement.pdf")
ruchi_feb = parse_tally_pdf("../data/ruchi 15kg po tin feb statement.pdf")

# Fix customer names in all
for df in [fortune_feb, ruchi_jan, ruchi_feb]:
    df['customer'] = df['customer'].apply(lambda x: re.sub(r'^\d{1,2}-\w{3}-\d{2}\s*', '', x).strip())

# Add SKU label
fortune_feb['sku'] = 'Fortune R.Oil 15KG'
ruchi_jan['sku'] = 'Ruchi Palm Oil 15KG'
ruchi_feb['sku'] = 'Ruchi Palm Oil 15KG'

# Combine into one master dataframe
master = pd.concat([fortune_feb, ruchi_jan, ruchi_feb], ignore_index=True)

print(f"Total transactions: {len(master)}")
print(f"Fortune Feb: {len(fortune_feb)} rows")
print(f"Ruchi Jan: {len(ruchi_jan)} rows")
print(f"Ruchi Feb: {len(ruchi_feb)} rows")
print(f"\nUnique customers: {master['customer'].nunique()}")
print(f"\nCustomer list:")
print(sorted(master['customer'].unique()))

Total transactions: 375
Fortune Feb: 72 rows
Ruchi Jan: 164 rows
Ruchi Feb: 139 rows

Unique customers: 32

Customer list:
['B R B JODHPUR SWEETS', 'Cash Customer', 'GANPATI SWEETS', 'GAYATRI SWEETS & CHATS', 'GOKUL SWEETS', 'Guru Shree Impex', 'Ivistaz Ecom Services Pvt Ltd', 'KAGGIS BAKERY', 'KARNATAKA OIL STORES', 'KONARK VEG RESTAURANT', 'M M FOOD PRODUCTS', 'MANJUNATHA AGENCIES', 'MANJUNATHA ENTERPRISES', 'Maan Sammaan Caterers', 'N ASHOK AITHAL', 'NAMMA FILTER COFFEE', 'NANDINI HOT CHIPS', 'NEW GAJANAND SWEETS', 'OM BIKNER SWEETS', 'RAKSHA PIPES PVT LTD', 'SHREE MILK SUPPLY', 'SHREE UDAY SWEETS', 'SHRI SAI SHYAM FOOD VENTURES', 'SOUTH VEG THINDIES', 'SREE NARASIMHA HOTEL PVT LTD-2', 'SRI SAI FOODS', 'SRI SIDDIVINAYAKA VENTURES {VEG FEAST}', 'Sangam Sweets', 'Sri Padmashree Trading Company', 'THE KAGGIS', 'TRIVIKRAMA ENTERPRISES', 'U D ROTIGHAR']


In [10]:
# Focus only on sales (not purchases)
sales = master[master['type'].isin(['Credit Sale', 'Cash Sale'])].copy()

# Convert date to datetime
sales['date'] = pd.to_datetime(sales['date'], format='%d-%b-%y')

# Focus on named customers only (not cash)
named = sales[sales['customer'] != 'Cash Customer'].copy()

# Count orders per customer
customer_summary = named.groupby('customer').agg(
    total_orders=('date', 'count'),
    first_order=('date', 'min'),
    last_order=('date', 'max'),
).reset_index()

customer_summary['days_span'] = (customer_summary['last_order'] - customer_summary['first_order']).dt.days
customer_summary['avg_days_between_orders'] = (customer_summary['days_span'] / customer_summary['total_orders']).round(1)

# Sort by most frequent
customer_summary = customer_summary.sort_values('total_orders', ascending=False)

print("=== YOUR PILOT CUSTOMERS — ORDER FREQUENCY ===\n")
print(customer_summary[['customer', 'total_orders', 'avg_days_between_orders', 'first_order', 'last_order']].to_string(index=False))

=== YOUR PILOT CUSTOMERS — ORDER FREQUENCY ===

                              customer  total_orders  avg_days_between_orders first_order last_order
                          U D ROTIGHAR            16                      3.2  2026-01-05 2026-02-26
                         Sangam Sweets             7                      3.1  2026-02-04 2026-02-26
                            THE KAGGIS             6                      8.0  2026-01-07 2026-02-24
                     SHREE UDAY SWEETS             3                     13.0  2026-01-15 2026-02-23
                        GANPATI SWEETS             3                     16.7  2026-01-05 2026-02-24
        SREE NARASIMHA HOTEL PVT LTD-2             3                      8.0  2026-02-02 2026-02-26
                    SOUTH VEG THINDIES             3                      9.3  2026-01-17 2026-02-14
                GAYATRI SWEETS & CHATS             3                      7.3  2026-01-19 2026-02-10
          SHRI SAI SHYAM FOOD VENTURES     

In [11]:
from datetime import timedelta

print("=== NEXT ORDER PREDICTIONS ===\n")

top_customers = ['U D ROTIGHAR', 'Sangam Sweets', 'THE KAGGIS', 
                 'SHREE UDAY SWEETS', 'GANPATI SWEETS',
                 'SREE NARASIMHA HOTEL PVT LTD-2',
                 'SRI SIDDIVINAYAKA VENTURES {VEG FEAST}']

for customer in top_customers:
    cust_data = customer_summary[customer_summary['customer'] == customer].iloc[0]
    last_order = cust_data['last_order']
    avg_cycle = cust_data['avg_days_between_orders']
    next_order = last_order + timedelta(days=avg_cycle)
    
    print(f"{customer}")
    print(f"  Last order: {last_order.strftime('%d %b %Y')}")
    print(f"  Avg cycle:  every {avg_cycle} days")
    print(f"  Next order: ~{next_order.strftime('%d %b %Y')}")
    print()

=== NEXT ORDER PREDICTIONS ===

U D ROTIGHAR
  Last order: 26 Feb 2026
  Avg cycle:  every 3.2 days
  Next order: ~01 Mar 2026

Sangam Sweets
  Last order: 26 Feb 2026
  Avg cycle:  every 3.1 days
  Next order: ~01 Mar 2026

THE KAGGIS
  Last order: 24 Feb 2026
  Avg cycle:  every 8.0 days
  Next order: ~04 Mar 2026

SHREE UDAY SWEETS
  Last order: 23 Feb 2026
  Avg cycle:  every 13.0 days
  Next order: ~08 Mar 2026

GANPATI SWEETS
  Last order: 24 Feb 2026
  Avg cycle:  every 16.7 days
  Next order: ~12 Mar 2026

SREE NARASIMHA HOTEL PVT LTD-2
  Last order: 26 Feb 2026
  Avg cycle:  every 8.0 days
  Next order: ~06 Mar 2026

SRI SIDDIVINAYAKA VENTURES {VEG FEAST}
  Last order: 19 Feb 2026
  Avg cycle:  every 7.5 days
  Next order: ~26 Feb 2026

